# Unilever Global — Full Lakehouse Router

This notebook is designed for the current Fabric flow:

`Mapping + Zip_copy -> Notebook1 -> Route_To_Lakehouses`

It reads:
- the mapping workbook from the **Configuration** Lakehouse;
- staged files created by `Notebook1`;
- category-to-Lakehouse relationships from the `Category` sheet;
- destination ABFS paths from the `LakeHouse mapping` sheet.

## Why the LakeHouse mapping sheet is still required

The `Category` sheet tells us **which logical Lakehouse a category belongs to**.

The `LakeHouse mapping` sheet tells us **where that Lakehouse physically lives in OneLake** through its `ABFS Path`.

Normalization only makes names comparable, for example:

- `HairCare` == `Hair Care`
- `HomeHygiene` == `Home & Hygiene`
- `DeodorantsFragrancesOralCareMasterbrandPersonalCare`
  == `Deodorants & Fragrances - Oral Care - Masterbrand Personal Care`

Normalization does **not** provide an ABFS destination. The `LakeHouse mapping` sheet remains the destination registry.

## Expected staging hierarchy

`Files/UL_Router/ReadyForSharePoint1/<Country>/<Category>/<Data Type>/<File>`

Example:

`Files/UL_Router/ReadyForSharePoint1/Brazil/Hair Care/Case Data/UBBRHSOF_CASEDATA_202301_202631.csv`

## Workbook structure

### Sheet: `Category`
Required columns:
- `SUB CATEGORY`
- `Dashboard Category`
- `Lakehouse_Key`

`SUB CATEGORY` cells may contain multiple values separated by line breaks.

### Sheet: `LakeHouse mapping`
Required columns:
- `Dashboard / Lakehouse`
- `ABFS Path`

There is **no requirement** for a `Lakehouse_Key` column in this sheet.

## Category resolution

A source Category can resolve through any of these forms:

1. Direct `Dashboard / Lakehouse` name
2. `SUB CATEGORY`
3. `Dashboard Category`
4. A normalized `Lakehouse_Key`

This supports both:
- `Oral Care`
- `Oral Care & Personal Care`
- `Deodorants & Fragrances - Oral Care - Masterbrand Personal Care`

## Destination hierarchy

`<Target Lakehouse>/Files/<Country>/<Category>/<Data Type>/<File>`

The original source Category folder name is preserved.

## Replacement rule

The full filename is compared **exactly, including week numbers**.

- Same exact filename -> replace existing file.
- Different week number -> different filename -> keep both.


In [13]:
# ============================================================
# PARAMETERS
# ============================================================
# IMPORTANT:
# Set this notebook's DEFAULT LAKEHOUSE to: Configuration

# If blank, the notebook tries to auto-find one Mapping*.xlsx file
# under /lakehouse/default/Files.
MAPPING_FILE = "/lakehouse/default/Files/UL_Router/Configuration/Mapping.xlsx"

# Staging folder produced by Notebook1
SOURCE_ROOT = "Files/UL_Router/GDF/ReadyForSharePoint1"

CATEGORY_SHEET = "Category"
LAKEHOUSE_SHEET = "LakeHouse mapping"

SUBCATEGORY_COLUMN = "SUB CATEGORY"
DASHBOARD_CATEGORY_COLUMN = "Dashboard Category"
CATEGORY_KEY_COLUMN = "Lakehouse_Key"

LAKEHOUSE_NAME_COLUMN = "Dashboard / Lakehouse"
ABFS_COLUMN = "ABFS Path"

# Safety options
DRY_RUN = False
DELETE_SOURCE_AFTER_SUCCESS = False
VALIDATE_ABFS_ROOTS = True

# Optional one-country test.
# Example: TEST_COUNTRY = "Brazil"
# Leave blank to process all countries.
TEST_COUNTRY = ""

# Audit location inside Configuration Lakehouse
AUDIT_DIR = "Files/UL_Router/GDF/Audit/LakehouseRouter"

StatementMeta(, 79d99950-b940-46d6-bd9c-ae9d6ca3f70e, 15, Finished, Available, Finished, False)

In [14]:
# ============================================================
# IMPORTS + HELPERS
# ============================================================

import os
import re
import json
import uuid
from datetime import datetime, timezone

import pandas as pd
import notebookutils


def normalize(value):
    """
    Normalize text for comparisons.

    Examples:
      HairCare -> haircare
      Hair Care -> haircare
      Home & Hygiene -> homehygiene
    """
    if value is None:
        return ""
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


def clean_text(value):
    if value is None:
        return ""
    text = str(value).replace("\xa0", " ").strip()
    text = re.sub(r"[ \t]+", " ", text)
    return text


def clean_abfs(value):
    """
    Remove display line breaks caused by Excel wrapping/pasting.
    """
    if value is None:
        return ""
    return (
        str(value)
        .replace("\r", "")
        .replace("\n", "")
        .strip()
        .rstrip("/")
    )


def join_path(base, *parts):
    return base.rstrip("/") + "/" + "/".join(
        str(part).strip("/")
        for part in parts
        if part is not None and str(part) != ""
    )


def walk_remote(root):
    """
    Recursively walk a Lakehouse/ABFS path with notebookutils.fs.
    Yields:
      file_info, relative_parts
    """
    def _walk(current, relative_parts):
        for item in notebookutils.fs.ls(current):
            name = item.name.rstrip("/")

            if item.isDir:
                yield from _walk(
                    item.path,
                    relative_parts + [name]
                )

            elif item.isFile:
                yield item, relative_parts + [name]

    yield from _walk(root, [])


def walk_local(root):
    for current, _, files in os.walk(root):
        for filename in files:
            yield os.path.join(current, filename)


def get_remote_size(path):
    parent, filename = path.rsplit("/", 1)

    for item in notebookutils.fs.ls(parent):
        if (
            item.isFile
            and item.name.rstrip("/") == filename
        ):
            return item.size

    return None

StatementMeta(, 79d99950-b940-46d6-bd9c-ae9d6ca3f70e, 16, Finished, Available, Finished, False)

In [15]:
# ============================================================
# FIND MAPPING WORKBOOK
# ============================================================

if not MAPPING_FILE:
    candidates = []

    for path in walk_local("/lakehouse/default/Files"):
        filename = os.path.basename(path).lower()

        if (
            filename.endswith(".xlsx")
            and "mapping" in filename
        ):
            candidates.append(path)

    exact = [
        p for p in candidates
        if os.path.basename(p).lower() == "mapping.xlsx"
    ]

    if len(exact) == 1:
        MAPPING_FILE = exact[0]

    elif len(candidates) == 1:
        MAPPING_FILE = candidates[0]

    else:
        print("Mapping workbook candidates:")
        for p in candidates:
            print("  -", p)

        raise RuntimeError(
            "Could not uniquely find the mapping workbook. "
            "Set MAPPING_FILE in the parameter cell to the exact "
            "/lakehouse/default/Files/...xlsx path."
        )

if not os.path.exists(MAPPING_FILE):
    raise FileNotFoundError(
        f"Mapping workbook not found: {MAPPING_FILE}"
    )

print("Using mapping workbook:")
print(MAPPING_FILE)

StatementMeta(, 79d99950-b940-46d6-bd9c-ae9d6ca3f70e, 17, Finished, Available, Finished, False)

Using mapping workbook:
/lakehouse/default/Files/UL_Router/Configuration/Mapping.xlsx


In [16]:
# ============================================================
# READ WORKBOOK + VALIDATE REQUIRED COLUMNS
# ============================================================

category_df = pd.read_excel(
    MAPPING_FILE,
    sheet_name=CATEGORY_SHEET,
    dtype=str
).fillna("")

lakehouse_df = pd.read_excel(
    MAPPING_FILE,
    sheet_name=LAKEHOUSE_SHEET,
    dtype=str
).fillna("")

required_category_columns = {
    SUBCATEGORY_COLUMN,
    DASHBOARD_CATEGORY_COLUMN,
    CATEGORY_KEY_COLUMN,
}

required_lakehouse_columns = {
    LAKEHOUSE_NAME_COLUMN,
    ABFS_COLUMN,
}

missing_category = (
    required_category_columns
    - set(category_df.columns)
)

missing_lakehouse = (
    required_lakehouse_columns
    - set(lakehouse_df.columns)
)

if missing_category:
    raise ValueError(
        f"Sheet '{CATEGORY_SHEET}' is missing: "
        f"{sorted(missing_category)}\n"
        f"Found: {list(category_df.columns)}"
    )

if missing_lakehouse:
    raise ValueError(
        f"Sheet '{LAKEHOUSE_SHEET}' is missing: "
        f"{sorted(missing_lakehouse)}\n"
        f"Found: {list(lakehouse_df.columns)}"
    )

print("Workbook loaded successfully.")
print("Category rows:", len(category_df))
print("LakeHouse mapping rows:", len(lakehouse_df))

StatementMeta(, 79d99950-b940-46d6-bd9c-ae9d6ca3f70e, 18, Finished, Available, Finished, False)

ValueError: Worksheet named 'LakeHouse mapping' not found

In [ ]:
# ============================================================
# BUILD CATEGORY LOOKUPS
# ============================================================

subcategory_to_key = {}
dashboard_to_key = {}
mapping_conflicts = []

for _, row in category_df.iterrows():

    lakehouse_key = clean_text(
        row[CATEGORY_KEY_COLUMN]
    )

    if not lakehouse_key:
        continue

    # --------------------------------------------------------
    # SUB CATEGORY
    # One Excel cell may contain multiple categories separated
    # by line breaks. Never split on commas because commas can
    # legitimately exist in a category name.
    # --------------------------------------------------------

    raw_subcategories = str(
        row[SUBCATEGORY_COLUMN] or ""
    )

    subcategories = [
        clean_text(x)
        for x in re.split(
            r"[\r\n]+",
            raw_subcategories
        )
        if clean_text(x)
    ]

    for subcategory in subcategories:
        n = normalize(subcategory)

        if (
            n in subcategory_to_key
            and subcategory_to_key[n]
            != lakehouse_key
        ):
            mapping_conflicts.append(
                f"SUB CATEGORY '{subcategory}' maps to both "
                f"'{subcategory_to_key[n]}' and "
                f"'{lakehouse_key}'"
            )

        else:
            subcategory_to_key[n] = (
                lakehouse_key
            )

    # --------------------------------------------------------
    # DASHBOARD CATEGORY
    # --------------------------------------------------------

    dashboard_category = clean_text(
        row[DASHBOARD_CATEGORY_COLUMN]
    )

    if dashboard_category:
        n = normalize(dashboard_category)

        if (
            n in dashboard_to_key
            and dashboard_to_key[n]
            != lakehouse_key
        ):
            mapping_conflicts.append(
                f"Dashboard Category "
                f"'{dashboard_category}' maps to both "
                f"'{dashboard_to_key[n]}' and "
                f"'{lakehouse_key}'"
            )

        else:
            dashboard_to_key[n] = (
                lakehouse_key
            )

if mapping_conflicts:
    print("CATEGORY MAPPING CONFLICTS:")

    for conflict in mapping_conflicts:
        print("  -", conflict)

    raise RuntimeError(
        "Conflicting category mappings exist."
    )

print(
    "SUB CATEGORY mappings:",
    len(subcategory_to_key)
)

print(
    "Dashboard Category mappings:",
    len(dashboard_to_key)
)

StatementMeta(, 79d99950-b940-46d6-bd9c-ae9d6ca3f70e, -1, Cancelled, , Cancelled, True)

In [ ]:
# ============================================================
# BUILD LAKEHOUSE DESTINATION REGISTRY
# Dashboard / Lakehouse -> ABFS Path
# ============================================================

lakehouse_to_abfs = {}
destination_conflicts = []

for _, row in lakehouse_df.iterrows():

    display_name = clean_text(
        row[LAKEHOUSE_NAME_COLUMN]
    )

    abfs = clean_abfs(
        row[ABFS_COLUMN]
    )

    # Ignore fully blank rows.
    if not display_name and not abfs:
        continue

    if not display_name or not abfs:
        raise ValueError(
            "Incomplete LakeHouse mapping row:\n"
            f"Dashboard / Lakehouse = '{display_name}'\n"
            f"ABFS Path = '{abfs}'"
        )

    if not abfs.lower().startswith("abfss://"):
        raise ValueError(
            f"ABFS path for '{display_name}' "
            f"does not begin with abfss://\n"
            f"Value: {abfs}"
        )

    normalized_name = normalize(
        display_name
    )

    if (
        normalized_name
        in lakehouse_to_abfs
        and lakehouse_to_abfs[
            normalized_name
        ]["abfs"] != abfs
    ):
        destination_conflicts.append(
            f"'{display_name}' has more than "
            f"one ABFS path."
        )

    else:
        lakehouse_to_abfs[
            normalized_name
        ] = {
            "display_name": display_name,
            "abfs": abfs,
        }

if destination_conflicts:
    raise RuntimeError(
        "Lakehouse destination conflicts:\n- "
        + "\n- ".join(
            destination_conflicts
        )
    )

print(
    "Lakehouse destination mappings:",
    len(lakehouse_to_abfs)
)

for record in sorted(
    lakehouse_to_abfs.values(),
    key=lambda x: x["display_name"].lower()
):
    print(
        f"  {record['display_name']} "
        f"-> {record['abfs']}"
    )

StatementMeta(, 79d99950-b940-46d6-bd9c-ae9d6ca3f70e, -1, Cancelled, , Cancelled, True)

In [ ]:
# ============================================================
# RESOLVE CATEGORY -> DESTINATION
# ============================================================

def get_target_from_key(lakehouse_key):
    """
    Lakehouse_Key is matched against Dashboard / Lakehouse
    using normalization.

    Example:
      HairCare
      Hair Care
    both normalize to:
      haircare
    """
    target = lakehouse_to_abfs.get(
        normalize(lakehouse_key)
    )

    if not target:
        raise ValueError(
            f"Lakehouse_Key '{lakehouse_key}' "
            f"does not match any "
            f"'Dashboard / Lakehouse' value."
        )

    return target


def resolve_category(category):
    """
    Resolution order:

    1. Direct Dashboard / Lakehouse name
    2. SUB CATEGORY -> Lakehouse_Key
    3. Dashboard Category -> Lakehouse_Key
    4. Normalized source category itself as a Lakehouse_Key
    """

    ncategory = normalize(category)

    # 1. Direct Dashboard / Lakehouse match
    if ncategory in lakehouse_to_abfs:
        target = lakehouse_to_abfs[
            ncategory
        ]

        return {
            "resolution": (
                "DIRECT_DASHBOARD_LAKEHOUSE"
            ),
            "lakehouse_key": (
                normalize(
                    target["display_name"]
                )
            ),
            "target": target,
        }

    # 2. SUB CATEGORY
    if ncategory in subcategory_to_key:
        key = subcategory_to_key[
            ncategory
        ]

        return {
            "resolution": "SUB_CATEGORY",
            "lakehouse_key": key,
            "target": get_target_from_key(
                key
            ),
        }

    # 3. Dashboard Category
    if ncategory in dashboard_to_key:
        key = dashboard_to_key[
            ncategory
        ]

        return {
            "resolution": (
                "DASHBOARD_CATEGORY"
            ),
            "lakehouse_key": key,
            "target": get_target_from_key(
                key
            ),
        }

    # 4. Category itself may already be a key
    target = lakehouse_to_abfs.get(
        ncategory
    )

    if target:
        return {
            "resolution": (
                "NORMALIZED_KEY"
            ),
            "lakehouse_key": category,
            "target": target,
        }

    raise ValueError(
        f"No mapping found for "
        f"source category '{category}'."
    )

StatementMeta(, 79d99950-b940-46d6-bd9c-ae9d6ca3f70e, -1, Cancelled, , Cancelled, True)

In [ ]:
# ============================================================
# VALIDATE ALL Lakehouse_Key VALUES
# ============================================================

key_errors = []

all_keys = sorted(
    set(subcategory_to_key.values())
    | set(dashboard_to_key.values())
)

for key in all_keys:
    if (
        normalize(key)
        not in lakehouse_to_abfs
    ):
        key_errors.append(
            f"Lakehouse_Key '{key}' does not "
            f"match a 'Dashboard / Lakehouse' "
            f"value after normalization."
        )

if key_errors:
    print(
        "Lakehouse_Key validation errors:"
    )

    for error in key_errors:
        print("  -", error)

    print(
        "\nAvailable Dashboard / Lakehouse "
        "values:"
    )

    for value in sorted(
        record["display_name"]
        for record
        in lakehouse_to_abfs.values()
    ):
        print("  -", value)

    raise RuntimeError(
        "Fix the Lakehouse_Key values "
        "shown above."
    )

print(
    "Every Lakehouse_Key maps to a "
    "LakeHouse mapping row."
)

StatementMeta(, 79d99950-b940-46d6-bd9c-ae9d6ca3f70e, -1, Cancelled, , Cancelled, True)

In [ ]:
# ============================================================
# OPTIONAL: VALIDATE ABFS ROOTS BEFORE COPYING
# ============================================================

if VALIDATE_ABFS_ROOTS:

    abfs_errors = []

    for record in sorted(
        lakehouse_to_abfs.values(),
        key=lambda x: x[
            "display_name"
        ].lower()
    ):
        display_name = record[
            "display_name"
        ]

        abfs = record["abfs"]

        try:
            exists = (
                notebookutils.fs.exists(
                    abfs
                )
            )

            if not exists:
                abfs_errors.append(
                    f"Destination does not exist "
                    f"or cannot be accessed: "
                    f"{display_name} -> {abfs}"
                )

            else:
                print(
                    f"OK: {display_name}"
                )

        except Exception as exc:
            abfs_errors.append(
                f"{display_name} -> "
                f"{abfs} -> {exc}"
            )

    if abfs_errors:
        print(
            "\nABFS DESTINATION ERRORS:"
        )

        for error in abfs_errors:
            print("  -", error)

        raise RuntimeError(
            "One or more destination ABFS "
            "paths are invalid or inaccessible. "
            "Use the exact 'Copy ABFS path' "
            "value from each target Lakehouse."
        )

    print(
        "\nAll destination ABFS roots "
        "are accessible."
    )

StatementMeta(, 79d99950-b940-46d6-bd9c-ae9d6ca3f70e, -1, Cancelled, , Cancelled, True)

In [ ]:
# ============================================================
# BUILD FULL ROUTE PLAN
#
# This cell DOES NOT COPY anything.
# It validates all staged files first.
# ============================================================

if not notebookutils.fs.exists(
    SOURCE_ROOT
):
    raise FileNotFoundError(
        f"Source staging root not found: "
        f"{SOURCE_ROOT}"
    )

source_files = list(
    walk_remote(
        SOURCE_ROOT
    )
)

if not source_files:
    raise RuntimeError(
        f"No files found under "
        f"{SOURCE_ROOT}"
    )

routes = []
route_errors = []

for file_info, parts in source_files:

    # Expected:
    # Country / Category / Data Type / File
    if len(parts) != 4:
        route_errors.append(
            f"Unexpected hierarchy: "
            f"{'/'.join(parts)}. "
            f"Expected "
            f"Country/Category/Data Type/File."
        )

        continue

    (
        country,
        category,
        data_type,
        filename,
    ) = parts

    if (
        TEST_COUNTRY
        and normalize(country)
        != normalize(TEST_COUNTRY)
    ):
        continue

    try:
        resolved = resolve_category(
            category
        )

        target = resolved[
            "target"
        ]

        # Preserve required destination:
        # Country -> Category -> Data Type
        target_folder = join_path(
            target["abfs"],
            country,
            category,
            data_type,
        )

        target_file = join_path(
            target_folder,
            filename,
        )

        routes.append({
            "source": (
                file_info.path
            ),
            "source_size": (
                file_info.size
            ),
            "country": country,
            "category": category,
            "data_type": data_type,
            "filename": filename,
            "resolution": (
                resolved[
                    "resolution"
                ]
            ),
            "lakehouse_key": (
                resolved[
                    "lakehouse_key"
                ]
            ),
            "target_lakehouse": (
                target[
                    "display_name"
                ]
            ),
            "target_folder": (
                target_folder
            ),
            "target_file": (
                target_file
            ),
        })

    except Exception as exc:
        route_errors.append(
            f"{'/'.join(parts)} "
            f"-> {exc}"
        )

if route_errors:

    print(
        "ROUTING VALIDATION ERRORS:"
    )

    for error in route_errors:
        print("  -", error)

    raise RuntimeError(
        f"{len(route_errors)} routing "
        f"error(s). Nothing copied."
    )

if not routes:
    raise RuntimeError(
        "No files remain after filters."
    )

print(
    f"Validated {len(routes)} files.\n"
)

for route in routes[:100]:

    print(
        f"{route['country']} | "
        f"{route['category']} | "
        f"{route['data_type']} | "
        f"{route['filename']} "
        f"-> "
        f"{route['target_lakehouse']} "
        f"[{route['resolution']}]"
    )

StatementMeta(, 79d99950-b940-46d6-bd9c-ae9d6ca3f70e, -1, Cancelled, , Cancelled, True)

In [ ]:
# ============================================================
# COPY / EXACT-FILENAME REPLACEMENT
# ============================================================
#
# WEEK NUMBER RULE:
#
# Existing:
#   ABC_202631.csv
# Incoming:
#   ABC_202631.csv
# -> same exact filename -> REPLACE
#
# Existing:
#   ABC_202631.csv
# Incoming:
#   ABC_202632.csv
# -> different filename -> KEEP BOTH
# ============================================================

run_id = (
    datetime.now(
        timezone.utc
    ).strftime(
        "%Y%m%dT%H%M%SZ"
    )
    + "_"
    + uuid.uuid4().hex[:8]
)

results = []

for index, route in enumerate(
    routes,
    start=1
):

    src = route["source"]
    dst_folder = (
        route["target_folder"]
    )
    dst = route["target_file"]

    existed_before = (
        notebookutils.fs.exists(
            dst
        )
    )

    operation = (
        "REPLACE_EXACT_FILENAME"
        if existed_before
        else "COPY_NEW_FILE"
    )

    print(
        f"[{index}/{len(routes)}] "
        f"{operation}\n"
        f"  Source: {src}\n"
        f"  Target: {dst}"
    )

    if DRY_RUN:

        results.append({
            **route,
            "operation": operation,
            "status": "DRY_RUN",
        })

        continue

    temp_path = None

    try:

        created = (
            notebookutils.fs.mkdirs(
                dst_folder
            )
        )

        if (
            not created
            and not (
                notebookutils.fs.exists(
                    dst_folder
                )
            )
        ):
            raise RuntimeError(
                f"Could not create "
                f"target folder: "
                f"{dst_folder}"
            )

        # First copy to a temporary file.
        # The existing final file remains
        # untouched until the new copy succeeds.

        temp_name = (
            f".incoming_"
            f"{uuid.uuid4().hex}_"
            f"{route['filename']}"
        )

        temp_path = join_path(
            dst_folder,
            temp_name,
        )

        copied = (
            notebookutils.fs.cp(
                src,
                temp_path,
                recurse=False,
            )
        )

        if not copied:
            raise RuntimeError(
                "notebookutils.fs.cp "
                "returned False"
            )

        if not (
            notebookutils.fs.exists(
                temp_path
            )
        ):
            raise RuntimeError(
                "Temporary copied file "
                "does not exist."
            )

        source_size = (
            route["source_size"]
        )

        temp_size = (
            get_remote_size(
                temp_path
            )
        )

        if (
            source_size is not None
            and temp_size is not None
            and source_size
            != temp_size
        ):
            raise RuntimeError(
                f"Temporary size mismatch: "
                f"source={source_size}, "
                f"temp={temp_size}"
            )

        # Exact filename replacement happens here.
        moved = (
            notebookutils.fs.mv(
                temp_path,
                dst,
                create_path=True,
                overwrite=True,
            )
        )

        if not moved:
            raise RuntimeError(
                "notebookutils.fs.mv "
                "returned False"
            )

        if not (
            notebookutils.fs.exists(
                dst
            )
        ):
            raise RuntimeError(
                "Final destination file "
                "does not exist."
            )

        final_size = (
            get_remote_size(
                dst
            )
        )

        if (
            source_size is not None
            and final_size is not None
            and source_size
            != final_size
        ):
            raise RuntimeError(
                f"Final size mismatch: "
                f"source={source_size}, "
                f"target={final_size}"
            )

        if (
            DELETE_SOURCE_AFTER_SUCCESS
        ):
            notebookutils.fs.rm(
                src,
                recurse=False,
            )

        results.append({
            **route,
            "operation": operation,
            "status": "SUCCESS",
            "final_size": final_size,
        })

    except Exception as exc:

        # Best-effort temporary cleanup.
        try:
            if (
                temp_path
                and (
                    notebookutils.fs.exists(
                        temp_path
                    )
                )
            ):
                notebookutils.fs.rm(
                    temp_path,
                    recurse=False,
                )

        except Exception:
            pass

        results.append({
            **route,
            "operation": operation,
            "status": "FAILED",
            "error": str(exc),
        })

        print(
            "  FAILED:",
            exc
        )

StatementMeta(, 79d99950-b940-46d6-bd9c-ae9d6ca3f70e, -1, Cancelled, , Cancelled, True)

In [ ]:
# ============================================================
# AUDIT + PIPELINE RESULT
# ============================================================

successes = [
    x for x in results
    if x["status"] == "SUCCESS"
]

failures = [
    x for x in results
    if x["status"] == "FAILED"
]

dry_runs = [
    x for x in results
    if x["status"] == "DRY_RUN"
]

summary = {
    "runId": run_id,
    "mappingFile": MAPPING_FILE,
    "sourceRoot": SOURCE_ROOT,
    "testCountry": TEST_COUNTRY,
    "dryRun": DRY_RUN,
    "planned": len(routes),
    "success": len(successes),
    "failed": len(failures),
    "dryRunCount": len(dry_runs),
    "newFiles": sum(
        x["operation"]
        == "COPY_NEW_FILE"
        for x in successes
    ),
    "replacedExactFilename": sum(
        x["operation"]
        == "REPLACE_EXACT_FILENAME"
        for x in successes
    ),
    "replacementRule": (
        "Full filename comparison. "
        "Week numbers are included."
    ),
}

print(
    "\n========== SUMMARY =========="
)

print(
    json.dumps(
        summary,
        indent=2
    )
)

notebookutils.fs.mkdirs(
    AUDIT_DIR
)

audit_path = (
    f"{AUDIT_DIR}/"
    f"router_{run_id}.json"
)

notebookutils.fs.put(
    audit_path,
    json.dumps(
        {
            "summary": summary,
            "results": results,
        },
        indent=2,
        default=str,
    ),
    overwrite=True,
)

print(
    "\nAudit written to:"
)

print(
    audit_path
)

if failures:

    print(
        "\nFAILED FILES:"
    )

    for item in failures:
        print(
            f"  - {item['source']} "
            f"-> "
            f"{item.get('error')}"
        )

    raise RuntimeError(
        f"{len(failures)} file(s) "
        f"failed. "
        f"See audit: {audit_path}"
    )

notebookutils.notebook.exit(
    json.dumps(
        summary
    )
)

StatementMeta(, 79d99950-b940-46d6-bd9c-ae9d6ca3f70e, -1, Cancelled, , Cancelled, True)

## Pipeline setup

Use:

`Mapping + Zip_copy -> Notebook1 -> Route_To_Lakehouses`

1. Import this notebook into the workspace.
2. Set **Configuration** as its default Lakehouse.
3. Deactivate the old SharePoint `Processed` Copy activity.
4. Add a Notebook activity after `Notebook1`.
5. Name it `Route_To_Lakehouses`.
6. Point it to this notebook.
7. For the first test, set `TEST_COUNTRY = "Brazil"`.
8. Once verified, set `TEST_COUNTRY = ""`.

### Important ABFS note

The values in `LakeHouse mapping -> ABFS Path` are the physical destinations.

They must be valid, accessible OneLake ABFS paths. This notebook validates them before copying anything. If an ABFS path is wrong, the run stops before any file is moved.

In [ ]:
# home_root = "abfss://7fec2aac-e051-46bd-abda-2af1dbdc6cc9@onelake.dfs.fabric.microsoft.com/12589b4a-95e9-4bdd-844a-a313f991241e/Files"
 
# print("HOME ROOT EXISTS:", notebookutils.fs.exists(home_root))
# print(notebookutils.fs.ls(home_root))
 

StatementMeta(, 79d99950-b940-46d6-bd9c-ae9d6ca3f70e, -1, Cancelled, , Cancelled, True)

In [ ]:
# oral_root = "abfss://7fec2aac-e051-46bd-abda-2af1dbdc6cc9@onelake.dfs.fabric.microsoft.com/9fbd3a73-cd07-4542-a7cb-5632224ffbee/Files"
 
# print("ORAL ROOT EXISTS:", notebookutils.fs.exists(oral_root))
# print(notebookutils.fs.ls(oral_root))
 

In [ ]:
# import json
 
# audit_path = "/lakehouse/default/Files/UL_Router/Audit/LakehouseRouter/router_20260922T070343Z_130a8fa3.json"
 
# with open(audit_path, "r", encoding="utf-8") as f:
#     audit = json.load(f)
 
# failed = [
#     x for x in audit["results"]
#     if x.get("status") == "FAILED"
#     and x.get("category") == "Oral Care & Personal Care"
# ]
 
# print("Oral Care failed records:", len(failed))
 
# test = failed[0]
 
# source = test["source"]
# target_folder = test["target_folder"]
# target_file = test["target_file"]
 
# print("\nFILE:")
# print(test["filename"])
 
# print("\nSOURCE:")
# print(source)
 
# print("\nTARGET FOLDER:")
# print(target_folder)
 
# print("\nTARGET FILE:")
# print(target_file)
 
# print("\nSOURCE EXISTS       :", notebookutils.fs.exists(source))
# print("TARGET FOLDER EXISTS:", notebookutils.fs.exists(target_folder))
# print("TARGET FILE EXISTS  :", notebookutils.fs.exists(target_file))
 

In [ ]:
# test_file = target_folder.rstrip("/") + "/__WRITE_TEST__.txt"
 
# try:
#     result = notebookutils.fs.put(
#         test_file,
#         "Fabric router write test",
#         True
#     )
 
#     print("PUT RESULT :", result)
#     print("TEST EXISTS:", notebookutils.fs.exists(test_file))
 
# except Exception as e:
#     print("PUT FAILED:")
#     print(str(e))
 

StatementMeta(, 79d99950-b940-46d6-bd9c-ae9d6ca3f70e, -1, Cancelled, , Cancelled, True)

In [ ]:
# copy_test_file = (
#     target_folder.rstrip("/")
#     + "/__COPY_TEST__"
#     + "." + source.split(".")[-1]
# )
 
# print("SOURCE:")
# print(source)
 
# print("\nCOPY TEST TARGET:")
# print(copy_test_file)
 
# try:
#     result = notebookutils.fs.cp(
#         source,
#         copy_test_file
#     )
 
#     print("\nCP RESULT :", result)
#     print("COPY EXISTS:", notebookutils.fs.exists(copy_test_file))
 
# except Exception as e:
#     print("\nCP FAILED:")
#     print(str(e))
 

StatementMeta(, 79d99950-b940-46d6-bd9c-ae9d6ca3f70e, -1, Cancelled, , Cancelled, True)

In [ ]:
# print("SOURCE:")
# print(source)
 
# print("\nEXACT TARGET:")
# print(target_file)
 
# print("\nBEFORE COPY:")
# print("Source exists:", notebookutils.fs.exists(source))
# print("Target exists:", notebookutils.fs.exists(target_file))
 
# try:
#     result = notebookutils.fs.cp(
#         source,
#         target_file
#     )
 
#     print("\nCP RESULT:", result)
#     print("TARGET EXISTS AFTER COPY:", notebookutils.fs.exists(target_file))
 
# except Exception as e:
#     print("\nEXACT COPY FAILED:")
#     print(str(e))
 

StatementMeta(, 79d99950-b940-46d6-bd9c-ae9d6ca3f70e, -1, Cancelled, , Cancelled, True)